# Training pipeline (torchgeo + torchvision)

Serge Wiltshire, 7/7/2026

Trains the single shared model from labeled counties of `TRAIN_STATE`, writes it to `outputs/models/{MODEL_NAME}/`, and uploads it to `s3://.../models/{MODEL_NAME}/` so every state pipeline (`pipeline_{STATE}.ipynb`) can use it.

Use the **Python (vrt)** kernel (see `setup_env.ipynb`).

## Setup

In [ ]:
from pathlib import Path
import sys
import os

project_root = Path.cwd().resolve().parent  # notebooks -> root
sys.path.insert(0, str(project_root / "src"))

from utils import states

TRAIN_STATE = "IN"                 # state whose labeled counties are used for training
MODEL_NAME = states.DEFAULT_MODEL  # -> outputs/models/{MODEL_NAME}/ (single-class mode appends _{FOCUS_CLASS})

# Default training counties (labels verified against TRAIN_STATE imagery)
training_counties = {
    'Benton', 'Boone', 'Brown', 'Carroll', 'Cass', 'Hamilton', 'Howard',
    'Jackson', 'Johnson', 'LaPorte', 'Miami', 'Montgomery',
    'Newton', 'Tipton', 'Wabash', 'Warren', 'White'
}

state_cfg = states.get_state(TRAIN_STATE)
county_data_dir = states.counties_dir(TRAIN_STATE)
unknown = sorted(training_counties - set(state_cfg.counties))
assert not unknown, f"not {TRAIN_STATE} counties: {unknown}"

# Training data / outputs
training_dir = states.training_data_dir()
training_dir.mkdir(parents=True, exist_ok=True)
vrt_path = training_dir / "multi_county_training_mosaic.vrt"
labels_path = training_dir / "multi_county_labels.gpkg"

print(f"Training state: {state_cfg.name} ({TRAIN_STATE}), canonical CRS {state_cfg.canonical_crs}")
print(f"Training counties ({len(training_counties)}): {sorted(training_counties)}")

## Fetch training inputs from S3

Imagery year pins (`{STATE}/training_county_imagery_years.csv`): each county's raw tile download is pinned to the imagery year its training labels were verified against.

In [ ]:
import subprocess

years_csv = states.imagery_years_csv(TRAIN_STATE)
years_csv.parent.mkdir(parents=True, exist_ok=True)
subprocess.check_call(["aws", "s3", "cp", states.s3_uri(states.s3_imagery_years_key(TRAIN_STATE)), str(years_csv)])

import pandas as pd
print(pd.read_csv(years_csv))

Fetch training labels (`{STATE}/training_labels/{county}/`) into `data/{STATE}/counties/{county}/`

In [ ]:
import subprocess

labels_prefix = states.s3_uri(states.s3_training_labels_prefix(TRAIN_STATE)) + "/"
county_data_dir.mkdir(parents=True, exist_ok=True)

listing = subprocess.check_output(["aws", "s3", "ls", labels_prefix], text=True)
label_counties = [
    line.split("PRE", 1)[1].strip().rstrip("/")
    for line in listing.splitlines()
    if line.strip().startswith("PRE")
]
print("found labeled counties:", label_counties)

for county in label_counties:
    subprocess.check_call([
        "aws", "s3", "sync",
        f"{labels_prefix}{county}/",
        str(states.county_dir(TRAIN_STATE, county)),
    ])

print("download complete")

## Canonical chips for training counties

Select counties to fetch rasters and reproject if needed

In [ ]:
import glob
from ipywidgets import SelectMultiple, Button, VBox
from IPython.display import display

all_counties = sorted([
    d for d in os.listdir(county_data_dir)
    if glob.glob(os.path.join(county_data_dir, d, "*.shp"))                   # must have at least one .shp in the county dir
])

cty_sel_fetch = SelectMultiple(
    options=all_counties,
    value=tuple(all_counties),  # default pick all
    description='counties:',
    rows=min(10, len(all_counties))
)

btn = Button(description="fetch/reproj these", button_style='success')

def confirm(_):
    print("selected counties:", list(cty_sel_fetch.value))

btn.on_click(confirm)
display(VBox([cty_sel_fetch, btn]))

Download imagery tiles, mosaic native-CRS groups, and crop canonical (seamlessly reprojected) chips - cached to S3

In [ ]:
from utils.prepare_reprojected_tiles import ensure_canonical_mosaic_for_counties

ensure_canonical_mosaic_for_counties(
    TRAIN_STATE,
    counties=list(cty_sel_fetch.value),
)

## Setup and Run Model Training

Select counties for training

In [ ]:
import glob
from ipywidgets import SelectMultiple, Button, VBox
from IPython.display import display

all_counties = sorted([
    d for d in os.listdir(county_data_dir)
    if os.path.isdir(os.path.join(county_data_dir, d, "canonical_tiles"))      # must have canonical_tiles/
    and glob.glob(os.path.join(county_data_dir, d, "*.shp"))                   # must have at least one .shp in the county dir
])

cty_sel_train = SelectMultiple(
    options=all_counties,
    value=tuple(c for c in all_counties if c in training_counties),
    description='counties:',
    rows=min(10, len(all_counties))
)

btn = Button(description="train on these", button_style='success')

def confirm(_):
    print("selected counties:", list(cty_sel_train.value))

btn.on_click(confirm)
display(VBox([cty_sel_train, btn]))

### Single-Class Training Configuration (Optional)

Toggle to train on a single focus class with tunable training splits:
- **Confirmed Positives**: `focus_class` with VerifiedTr=1 (verified examples)
- **In-Class Negatives**: `focus_class` with VerifiedTr=0 (misclassified examples—look like focus class but aren't)
- **Out-of-Class Negatives**: Features from other classes (fundamentally different, easier to distinguish)
- **Overlap removal**: Out-of-class features overlapping confirmed positives are automatically excluded to prevent conflicting training signals

Adjust `POSITIVE_RATIO` to control overall split (confirmed vs all negatives) and `IN_CLASS_RATIO` to control negative composition (in-class vs out-of-class).

Set `SINGLE_CLASS_MODE = False` for multi-class training (default).

In [ ]:
# =============================
# SINGLE-CLASS TRAINING TOGGLE + RATIOS
# =============================

# Set to True to enable single-class training
SINGLE_CLASS_MODE = False  # Change to True to enable

# If single-class, which class to focus on?
FOCUS_CLASS = 'Tile_Outlet'  # options: 'Bank_Erosion', 'Spillway', 'Culvert_Structure', 'Tile_Inlet', 'Tile_Outlet'

# INCLUDE_BACKGROUND_CLASS: whether single-class training includes a negative/"Background"
# class at all.
# - True (default): builds a Background pool sized via POSITIVE_RATIO/IN_CLASS_RATIO below
#   (existing tunable-background recipe).
# - False: keeps ONLY confirmed positives (FOCUS_CLASS with VerifiedTr=1) — no Background
#   class, no negatives at all. POSITIVE_RATIO/IN_CLASS_RATIO are ignored in this case.
INCLUDE_BACKGROUND_CLASS = True

# POSITIVE_RATIO: percentage of training data that should be confirmed positives
# Ignored when INCLUDE_BACKGROUND_CLASS = False.
# - 80 (default): 80% confirmed Tile_Outlet (VerifiedTr=1), 20% negatives (mix of in-class and out-of-class)
# - 50: 50/50 split between confirmed and negatives
# - 20: 20% confirmed, 80% negatives
POSITIVE_RATIO = 80

# IN_CLASS_RATIO: percentage of the negative pool that should be in-class negatives
# Ignored when INCLUDE_BACKGROUND_CLASS = False.
# - 50 (default): 50/50 split between in-class (VerifiedTr=0) and out-of-class (other classes)
# - 80: 80% in-class (misclassified examples that look like Tile_Outlet), 20% out-of-class
# - 20: 20% in-class, 80% out-of-class (emphasizes learning from truly different features)
# In-class negatives are "hard" cases for model to learn (similar appearance, wrong label)
IN_CLASS_RATIO = 50

# BACKGROUND_RATIO: multi-class mode only. Caps the relabeled Background pool
# (VerifiedTr=0 rows, across all 5 classes) so it's at most this % of the final combined
# dataset (confirmed real-class rows + kept Background rows). Without a cap, Background
# can end up dwarfing every real class (e.g. observed 65.8% of all instances), starving
# the rarest classes (Spillway, Tile_Inlet) of proportional training signal even with
# stratified sampling. Ignored in single-class mode.
BACKGROUND_RATIO = 50

# Display current mode
# NOTE: POSITIVE_RATIO/IN_CLASS_RATIO/INCLUDE_BACKGROUND_CLASS only apply in single-class
# mode (prepare_multicounty_training()'s single_class=True path). Multi-class mode trains
# on the natural label distribution of all 5 classes (plus unconfirmed VerifiedTr=0 rows
# relabeled to an explicit "Background" class) and ignores these knobs entirely, so they
# must NOT be printed as if they were in effect when SINGLE_CLASS_MODE = False.
mode_str = f"SINGLE-CLASS ({FOCUS_CLASS})" if SINGLE_CLASS_MODE else "MULTI-CLASS (all 5 classes)"
print(f"Training Mode: {mode_str}")

if not SINGLE_CLASS_MODE:
    print(f"\nMulti-class strategy: all 5 classes trained together on their natural label distribution.")
    print(f"  - POSITIVE_RATIO / IN_CLASS_RATIO / INCLUDE_BACKGROUND_CLASS do not apply in this mode")
    print(f"  - Unconfirmed (VerifiedTr=0) rows across all classes are relabeled to an explicit 'Background' class")
    print(f"  - Background is capped at ~{BACKGROUND_RATIO}% of the final combined dataset (BACKGROUND_RATIO)")
    print(f"  - classes.json / expected_classes include 'Background' as a 6th class in this mode")
elif not INCLUDE_BACKGROUND_CLASS:
    print(f"\nSingle-class strategy: POSITIVES-ONLY (no Background class)")
    print(f"  - Confirmed positives: 100% {FOCUS_CLASS} (VerifiedTr=1)")
    print(f"  - No negatives, no Background class (POSITIVE_RATIO/IN_CLASS_RATIO ignored)")
else:
    print(f"Positive Ratio: {POSITIVE_RATIO}% confirmed {FOCUS_CLASS}, {100-POSITIVE_RATIO}% negatives")
    print(f"Negative Composition: {IN_CLASS_RATIO}% in-class (VerifiedTr=0), {100-IN_CLASS_RATIO}% out-of-class (other classes)")
    print(f"\nSingle-class strategy:")
    print(f"  - Confirmed positives: {POSITIVE_RATIO}% {FOCUS_CLASS} (VerifiedTr=1)")
    print(f"  - Negatives: {100-POSITIVE_RATIO}% total")
    print(f"    - In-class: {IN_CLASS_RATIO}% × {100-POSITIVE_RATIO}% = {int(IN_CLASS_RATIO * (100-POSITIVE_RATIO) / 100)}% of training data")
    print(f"      ({FOCUS_CLASS} with VerifiedTr=0—misclassified examples)")
    print(f"    - Out-of-class: {100-IN_CLASS_RATIO}% × {100-POSITIVE_RATIO}% = {int((100-IN_CLASS_RATIO) * (100-POSITIVE_RATIO) / 100)}% of training data")
    print(f"      (Other classes—fundamentally different features)")
    print(f"\nOverlap removal: out-of-class features overlapping confirmed {FOCUS_CLASS} will be excluded")

Merge multi-county training feature labels into a gpkg, and imagery into a VRT

In [ ]:
from utils.prepare_multicounty_training import prepare_multicounty_training

prepare_multicounty_training(
    TRAIN_STATE,
    selected_counties=list(cty_sel_train.value),
    out_labels=str(labels_path),
    out_vrt=str(vrt_path),
    verified_only=True,
    debug_plots=False,
    debug_plot_mode="sample",  # "sample" = one tile per county, "all" = every canonical tile
    single_class=SINGLE_CLASS_MODE,
    focus_class=FOCUS_CLASS if SINGLE_CLASS_MODE else None,
    positive_ratio=POSITIVE_RATIO,
    in_class_ratio=IN_CLASS_RATIO,
    include_background=INCLUDE_BACKGROUND_CLASS,
    background_ratio=BACKGROUND_RATIO,
)

Validate Training Data (optional)

Checks performed:
- File and CRS validity
- Class presence and balance
- Geometry integrity and spatial overlap
- Per-county balance (flags suspicious counties with >95% single class)

Output interpretation:
- PASSED: Data is ready to train
- WARNINGS: Check items listed but can proceed
- ERRORS: Fix data and re-run prepare step before training

In [ ]:
import sys
sys.path.insert(0, str(project_root))
from scripts.validate_training_data import validate_training_data

print("\n" + "="*70)
print("VALIDATING TRAINING DATASET")
print("="*70)

# Determine expected classes based on training mode
if SINGLE_CLASS_MODE:
    if INCLUDE_BACKGROUND_CLASS:
        expected_classes = ['Background', FOCUS_CLASS]
        config_notes = f"Positive ratio: {POSITIVE_RATIO}% confirmed, {100-POSITIVE_RATIO}% negatives (in-class {IN_CLASS_RATIO}%, out-of-class {100-IN_CLASS_RATIO}%)"
    else:
        # Positives-only mode: prepare_multicounty_training() returns ONLY confirmed
        # focus_class rows (no Background) when include_background=False.
        expected_classes = [FOCUS_CLASS]
        config_notes = "Positives-only (no Background class, no negatives)"
    mode_desc = f"Single-class ({FOCUS_CLASS})"
else:
    # Multi-class mode: prepare_multicounty_training() relabels unconfirmed
    # (VerifiedTr=0) rows across all classes to an explicit "Background" class
    # (2026-08-11 fix), so it's a real expected class here too, matching
    # classes.json for multi-class mode.
    expected_classes = ['Bank_Erosion', 'Spillway', 'Culvert_Structure', 'Tile_Inlet', 'Tile_Outlet', 'Background']
    mode_desc = "Multi-class (all 5 classes + Background)"
    config_notes = "Multi-class mode (positive/in-class ratios do not apply; VerifiedTr=0 rows relabeled to Background)"

print(f"Mode: {mode_desc}")
print(f"Config: {config_notes}")
print(f"Expected classes: {expected_classes}\n")

is_valid, errors, warnings, summary = validate_training_data(
    raster_path=str(vrt_path),
    labels_path=str(labels_path),
    expected_classes=expected_classes,
    verbose=True
)

# Check results before training
if not is_valid:
    print("\nVALIDATION FAILED - Cannot proceed with training!")
    print(f"Errors: {errors}")
    print("\nFix data issues and re-run prepare_multicounty_training()")
    import sys; sys.exit(1)
    
if warnings:
    print(f"\n{len(warnings)} warnings detected:")
    for w in warnings:
        print(f"  - {w}")
else:
    print("\nNo warnings - data looks good!")

print(f"\nVALIDATION PASSED")
print(f"  Total labels: {summary['n_labels']}")
print(f"  Spatial coverage: {summary['spatial_coverage']*100:.1f}%")

# ========================================================================
# Inspect native pixel size of labeled features (for tile_size / anchor tuning)
# ========================================================================
import geopandas as gpd
import rasterio
import numpy as np

gdf_check = gpd.read_file(labels_path)

with rasterio.open(vrt_path) as src:
    px_w = abs(src.transform.a)
    px_h = abs(src.transform.e)

print(f"\nNative pixel size: {px_w:.4f} x {px_h:.4f} (CRS units per pixel)\n")

# Always inspect all classes (not just FOCUS_CLASS) so tile_size/anchor decisions
# account for larger multi-class features too (e.g. Culvert_Structure), even when
# currently training in single-class mode.
target_classes = ['Bank_Erosion', 'Spillway', 'Culvert_Structure', 'Tile_Inlet', 'Tile_Outlet']
for cls in target_classes:
    if cls == 'Background':
        continue
    subset = gdf_check[gdf_check['Classname'] == cls]
    if subset.empty:
        print(f"{cls}: no features found")
        continue

    bounds = subset.geometry.bounds
    width_px = (bounds['maxx'] - bounds['minx']) / px_w
    height_px = (bounds['maxy'] - bounds['miny']) / px_h
    diag_px = np.sqrt(width_px**2 + height_px**2)

    print(f"{cls} (n={len(subset)}):")
    print(f"  width  (px): min={width_px.min():.1f}  median={width_px.median():.1f}  mean={width_px.mean():.1f}  max={width_px.max():.1f}")
    print(f"  height (px): min={height_px.min():.1f}  median={height_px.median():.1f}  mean={height_px.mean():.1f}  max={height_px.max():.1f}")
    print(f"  diagonal (px): median={diag_px.median():.1f}  p90={np.percentile(diag_px, 90):.1f}  p95={np.percentile(diag_px, 95):.1f}\n")

Set feature class list according to training type

In [ ]:
import json

# Generate classes.json based on training mode
if SINGLE_CLASS_MODE:
    if INCLUDE_BACKGROUND_CLASS:
        # Single-class mode: include focus class + Background (generated from tunable hard/easy negatives)
        classes_list = [FOCUS_CLASS, 'Background']
        mode_label = f"SINGLE-CLASS ({FOCUS_CLASS})"
    else:
        # Positives-only mode: no explicit Background class (no negatives were sampled)
        classes_list = [FOCUS_CLASS]
        mode_label = f"SINGLE-CLASS ({FOCUS_CLASS}, positives-only)"
    classes_json_path = training_dir / f"classes_single_{FOCUS_CLASS.lower()}.json"
else:
    # Multi-class mode: prepare_multicounty_training() relabels unconfirmed (VerifiedTr=0)
    # rows across all classes to an explicit "Background" class, so it must be included
    # here as a real (6th) class, not left implicit.
    classes_list = ['Bank_Erosion', 'Spillway', 'Culvert_Structure', 'Tile_Inlet', 'Tile_Outlet', 'Background']
    classes_json_path = training_dir / "classes_w_background.json"
    mode_label = "MULTI-CLASS"

classes_json_path.parent.mkdir(parents=True, exist_ok=True)

# Write classes.json
classes_data = {'classes': classes_list}
with open(classes_json_path, 'w') as f:
    json.dump(classes_data, f, indent=2)

# Update classes_path for training
classes_path = classes_json_path

print(f"Mode: {mode_label}")
print(f"Classes: {classes_list}")
print(f"Classes JSON saved to: {classes_path}")

Make sure CUDA is available and GPUs are visible

In [ ]:
import torch, os
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
print("visible_devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


# to watch gpus in real time (from terminal):

# # 1) point-in-time snapshot
# nvidia-smi

# # 2) live, 1-second updates
# watch -n1 nvidia-smi

# # 3) process-focused view (top-like)
# nvidia-smi pmon -c 100 -s um

# to monitor CPUs
# top -H

# to monitor disk
# cat /proc/22106/io

Setup Training Parameters

In [ ]:
# Setup training output directory based on mode
if SINGLE_CLASS_MODE:
    train_model_name = f"{MODEL_NAME}_{FOCUS_CLASS}"
    mode_label = f"single-class ({FOCUS_CLASS})"
else:
    train_model_name = MODEL_NAME
    mode_label = "multi-class"
train_out_dir = states.model_dir(train_model_name)

import os
os.makedirs(train_out_dir, exist_ok=True)
print(f"Training mode: {mode_label}")
print(f"Output directory: {train_out_dir}\n")

# Training hyperparameters
config = {
    'task': 'instance_seg',  # or 'detection' for Faster RCNN
    'device': 'cuda',
    'tile_size': 512,  # kept at original 512 (not reduced to 256): the all-class
                       # pixel-size inspection (Validate Training Data cell) showed
                       # Culvert_Structure diagonals up to p95=400px / max width-height
                       # ~500px, which would be systematically clipped at a 256px tile.
    'epochs': 20,  # reverted to 20: model uses stock torchvision anchors + full
                   # pretrained detection weights again (RPN/box/mask heads no longer
                   # train from scratch), so the extra epochs added for scratch-trained
                   # heads are no longer needed.
    'batch_size': 2,
    'grad_accum': 1,
    'lr': 0.0002,
    'num_workers': 8,
    'prefetch_factor': 4,
    'normalize': 'imagenet',
    # max_per_class is a per-class cap on generated training windows
    # (tiling.py::make_label_centered_training_windows). Deliberately left effectively
    # UNCAPPED (100_000, far above any single class's raw label count) in both modes:
    # a real per-class cap (e.g. 500) truncates EVERY class down to that number once its
    # raw count exceeds it, which discards the majority of minority classes' own labels
    # too (not just the dominant class's) and provides no balancing benefit that
    # StratifiedWeightedSampler (below) doesn't already give via inverse-frequency
    # resampling. For a full multi-class run intended to get the best possible model,
    # keep this uncapped so every class's full label set is available as training
    # windows; StratifiedWeightedSampler handles the actual class-frequency rebalancing
    # at sampling time. NOTE: uncapped means ~9x more training windows than the old
    # 500-cap default, so expect proportionally longer epochs.
    'max_per_class': 100_000,
    'stratified_sampling': True,  # NEW: Use stratified sampling for multi-county
    'classes_json': str(classes_path),  # Classes with background
}

print("Training Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

### Generate Nohup Training Command

In [ ]:
from pathlib import Path

# Resolve absolute paths so the printed command works from any terminal's cwd
_abs_train_script = (project_root / "scripts" / "train.py").resolve()
_abs_vrt_path = Path(vrt_path).resolve()
_abs_labels_path = Path(labels_path).resolve()
_abs_train_out_dir = Path(train_out_dir).resolve()

# Built as a SINGLE physical line (no "\" line-continuations) so it pastes reliably
# into any terminal -- multi-line backslash-continued commands can silently break on
# paste (trailing whitespace after "\", bracketed-paste chunking, etc.), which causes
# nohup to lose its stdout/stdin redirection and the log file to never populate.
_train_args = (
    f"--task {config['task']} "
    f"--device {config['device']} "
    f'--raster_path "{_abs_vrt_path}" '
    f'--labels_path "{_abs_labels_path}" '
    f'--out_dir "{_abs_train_out_dir}" '
    f'--classes_json "{config["classes_json"]}" '
    f"--tile_size {config['tile_size']} "
    f"--epochs {config['epochs']} "
    f"--batch_size {config['batch_size']} "
    f"--grad_accum {config['grad_accum']} "
    f"--lr {config['lr']} "
    f"--num_workers {config['num_workers']} "
    f"--prefetch_factor {config['prefetch_factor']} "
    f"--normalize {config['normalize']} "
    f"--max_per_class {config['max_per_class']} "
    f"--stratified_sampling"
)

nohup_cmd = (
    f"cd {project_root.resolve()}; "
    f"export CUDA_VISIBLE_DEVICES=0,1,2,3; "
    f"export OMP_NUM_THREADS=4; "
    f"export MKL_NUM_THREADS=4; "
    f"export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; "
    f'nohup python -m torch.distributed.run --nproc_per_node=4 "{_abs_train_script}" {_train_args} '
    f'> "{_abs_train_out_dir}/train_$(date +%Y%m%d_%H%M%S).log" 2>&1 & disown'
)

print("Copy/paste the ONE LINE command below into a SageMaker Terminal tab (NOT this notebook) to")
print("run training detached from the Jupyter kernel, so it survives session timeouts:\n")
print(nohup_cmd)

### Run Training in Notebook

In [ ]:
import os

# Setup environment
%env CUDA_VISIBLE_DEVICES=0,1,2,3
%env OMP_NUM_THREADS=4
%env MKL_NUM_THREADS=4
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# Build training command
cmd = f"""python -m torch.distributed.run --nproc_per_node=4 \"{project_root / 'scripts' / 'train.py'}\" \
    --task {config['task']} \
    --device {config['device']} \
    --raster_path \"{vrt_path}\" \
    --labels_path \"{labels_path}\" \
    --out_dir \"{train_out_dir}\" \
    --classes_json \"{config['classes_json']}\" \
    --tile_size {config['tile_size']} \
    --epochs {config['epochs']} \
    --batch_size {config['batch_size']} \
    --grad_accum {config['grad_accum']} \
    --lr {config['lr']} \
    --num_workers {config['num_workers']} \
    --prefetch_factor {config['prefetch_factor']} \
    --normalize {config['normalize']} \
    --max_per_class {config['max_per_class']} \
    --stratified_sampling
"""

print("Running training with stratified sampling and background class...")
print(cmd)
print()

os.system(cmd)

## Upload the trained model to S3

Uploads `outputs/models/{model}/model_final.pth` and `classes.json` to `s3://.../models/{model}/`, where the state pipelines download it from. Refuses to overwrite a different existing model unless `overwrite=True`.

In [ ]:
from utils.model_store import upload_model

upload_model(Path(train_out_dir).name)